# Практическая работа № 2
## Иерархия игровых сущностей: ABC, протоколы, dataclass, миксины

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 2 — Углублённое ООП на Python  
**Продолжительность:** 4 ч аудиторно + 4 ч самостоятельно  
**Формат сдачи:** Git-репозиторий + тесты + README + ADR  
**Максимальный балл:** 5

---

## О чём эта работа

В первой практической работе вы провели рефакторинг процедурного кода в ООП и получили работающую игру. Но архитектура осталась «плоской»: нет единого базового класса для игровых объектов, способности не выделены в интерфейсы, конфигурация зашита магическими числами, логирование отсутствует.

Этот модуль — про **архитектурные инструменты**, которые превращают набор классов в расширяемую систему:

- **ABC** — явный контракт для игровых объектов;
- **Протоколы** — структурные интерфейсы для способностей;
- **dataclass** — конфигурации и компоненты без шаблонного кода;
- **Миксины** — переиспользование сквозной функциональности;
- **`__slots__`** — оптимизация памяти;
- **Дескрипторы** — управление доступом к атрибутам.

## Цели работы

После выполнения вы сможете:
- проектировать иерархию игровых сущностей с явными контрактами;
- выбирать между ABC и Protocol в зависимости от задачи;
- выносить конфигурацию в `@dataclass(frozen=True)`;
- применять миксины для логирования и сериализации;
- оптимизировать память через `__slots__`;
- писать юнит-тесты на контракты (ABC, протоколы, dataclass, миксины).

## План работы

| Шаг | Что делаем | Что изучаем |
|---|---|---|
| 0 | Настройка окружения | venv, pip, Git |
| 1 | Анализ текущей архитектуры | декомпозиция, выявление проблем |
| 2 | Проектирование ABC и протоколов | контракты, структурная типизация |
| 3 | Реализация `GameObject`, протоколов | ABC, `@abstractmethod`, `Protocol` |
| 4 | Конфигурации через `@dataclass` | `frozen`, `field(default_factory)` |
| 5 | Миксин логирования | множественное наследование, MRO |
| 6 | Интеграция в существующие классы | композиция, DI |
| 7 | Юнит-тесты | pytest, проверка контрактов |
| 8 | Документация и финальный коммит | README, ADR, PEP 8 |

**Важно:** этот блокнот содержит только задания. Готовые решения вы пишете сами. Если нужен образец — смотрите `practice2_game_objects.ipynb`, но **не копируйте его вслепую**: адаптируйте под свою тему проекта.

**Для углублённого изучения теории:**
- [Real Python: Implementing Interfaces in Python](https://realpython.com/python-interface/)
- [PEP 544 — Protocols](https://peps.python.org/pep-0544/)
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/)

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

for pkg in ["pytest", "pycodestyle"]:
    try:
        __import__(pkg)
        print(f"OK: {pkg}")
    except ImportError:
        print(f"MISSING: {pkg} — установите: pip install {pkg}")

---
# Шаг 0. Настройка окружения

## Зачем этот шаг

Мы продолжаем работать в том же репозитории, что и в задании №1. Новых зависимостей не требуется — `pytest` и `pycodestyle` уже установлены. Но нужно убедиться, что окружение активно и проект на месте.

## Задание 0.1

Откройте терминал в папке вашего проекта (созданного в задании №1). Убедитесь, что:

1. Активировано виртуальное окружение (`.venv`).
2. Git-репозиторий на месте (`git status` показывает чистоту).
3. Все тесты из задания №1 проходят (`pytest -v`).

**Если что-то не так** — вернитесь к заданию №1 и восстановите состояние. Не начинайте новую работу на сломанном фундаменте.

## Задание 0.2

Создайте ветку для новой работы:

```bash
git checkout -b module2-game-objects
```

**Зачем ветка:** это позволит экспериментировать, не ломая основную ветку. Если что-то пойдёт не так — можно вернуться к `main` и начать заново.

---
# Шаг 1. Анализ текущей архитектуры

## Зачем этот шаг

Прежде чем что-то менять, нужно понять, **что именно** нужно менять и **почему**. В задании №1 вы получили работающий код — но «работает» не значит «хорошо спроектирован». Этот шаг — аудит архитектуры.

## Теория: что искать

Ищите четыре категории проблем:

1. **Отсутствие единого контракта.** Есть ли базовый класс для всех игровых сущностей? Можно ли написать функцию, работающую с «любой сущностью»?
2. **Неявные способности.** Как вы проверяете, что объект умеет двигаться или отрисовываться? Через `hasattr`? Через `isinstance`? Это признак отсутствия протоколов.
3. **Магические числа.** Есть ли в коде значения вроде `10`, `100`, `20` без объяснения? Они дублируются? Их можно изменить без правки кода?
4. **Отсутствие логирования.** Если что-то идёт не так, как вы это диагностируете? `print()` — признак, что нужен миксин.

## Задание 1.1. Аудит

Пройдитесь по своим классам (`Detective`, `Div`, `Location`, `Game`, `Clue` или их аналогам) и заполните таблицу:

| Класс | Есть общий предок? | Есть магические числа? | Есть логирование? | Какие способности неявны? |
|---|---|---|---|---|
| ... | ... | ... | ... | ... |

**Сохраните в файл** `docs/analysis_module2.md` — это часть документации проекта.

## Задание 1.2. Проблемы и план

Ответьте письменно:

1. Какие классы должны наследоваться от `GameObject`? Почему?
2. Какие способности нужно выделить в протоколы? Приведите минимум 3.
3. Какие значения должны стать конфигурациями? Перечислите их.
4. Какие классы должны получить логирование через миксин?

**Сохраните в** `docs/analysis_module2.md`.

---
# Шаг 2. Проектирование ABC и протоколов

## Зачем этот шаг

Проектирование до кодирования экономит часы рефакторинга. Ошибка на диаграмме стоит 5 минут, ошибка в коде — часы.

## Теория: ABC vs Protocol

| Критерий | ABC | Protocol |
|---|---|---|
| Требует наследования | Да | Нет |
| Проверка во время выполнения | Да | С `@runtime_checkable` |
| Когда использовать | Базовый класс с общей логикой | Способность, роль |

**Правило:** ABC — для того, **чем объект является**. Protocol — для того, **что объект умеет**.

**Для углублённого изучения:**
- [Real Python: Interfaces in Python](https://realpython.com/python-interface/)
- [PEP 544](https://peps.python.org/pep-0544/)

## Задание 2.1. Диаграмма классов

Нарисуйте UML-диаграмму (Mermaid), включающую:

- **ABC `GameObject`** — базовый класс;
- **Протоколы `Movable`, `Drawable`, `Collidable`** — способности;
- **Ваши игровые сущности** (`Detective`, `Div`, `Location` и т.д.) — какие протоколы они реализуют;
- **Связи** между классами.

### Шаблон для заполнения

```mermaid
classDiagram
    class GameObject {
        <<abstract>>
        -_id: int
        -_name: str
        -_active: bool
        +update(dt) void
        +draw() str
    }
    class Movable {
        <<protocol>>
        +move(dx, dy) void
    }
    class Drawable {
        <<protocol>>
        +draw() str
    }
    class Collidable {
        <<protocol>>
        +collision_radius float
        +collides_with(other) bool
    }
    class ВашКласс1 {
        ...
    }
    GameObject <|-- ВашКласс1
    ВашКласс1 ..|> Drawable : реализует
```

**Сохраните в** `docs/uml_module2.md`.

## Задание 2.2. Ответьте письменно

1. Почему `GameObject` — это ABC, а не Protocol?
2. Почему `Movable` — это Protocol, а не ABC?
3. Какие ваши классы реализуют несколько протоколов одновременно? Почему?
4. Есть ли класс, который не должен наследоваться от `GameObject`? Обоснуйте.

---
# Шаг 3. Реализация `GameObject` и протоколов

## Зачем этот шаг

ABC и протоколы — фундамент всей дальнейшей архитектуры. Если они спроектированы правильно, всё остальное встанет на свои места.

## Теория: ключевые моменты

1. **ABC нельзя инстанцировать.** Python выбросит `TypeError` при попытке создать объект, если хотя бы один `@abstractmethod` не реализован.
2. **`@runtime_checkable`** нужен, если вы хотите использовать `isinstance()` с протоколом. Без него проверка работает только статически (mypy, IDE).
3. **`__slots__ = ()`** в протоколах и миксинах — хорошая практика, чтобы не создавать `__dict__`.

**Для углублённого изучения:**
- [Python docs: abc](https://docs.python.org/3/library/abc.html)
- [Real Python: Protocols](https://realpython.com/python-protocol/)

## Задание 3.1. Реализовать `GameObject` (ABC)

Создайте файл `core/game_object.py`:

**Требования к классу:**

1. Наследуется от `abc.ABC`.
2. Имеет автоматический счётчик `_next_id` для уникальных id.
3. В `__init__` принимает `name: str`.
4. Приватные атрибуты: `_id`, `_name`, `_active`.
5. Свойства только для чтения: `id`, `name`, `active`.
6. Метод `deactivate()` — помечает объект как неактивный.
7. Два абстрактных метода: `update(dt: float) -> None` и `draw() -> str`.
8. `__repr__` показывает класс, id и имя.

**Проверьте себя:**
- Попытка `GameObject("test")` → `TypeError`.
- Подкласс без `update` или `draw` → `TypeError`.
- Подкласс с обоими методами → создаётся, получает уникальный id.

## Задание 3.2. Реализовать протоколы

Создайте файл `core/protocols.py` с тремя протоколами:

**`Movable`** (методы):
- `move(dx: float, dy: float) -> None`

**`Drawable`** (методы):
- `draw() -> str`

**`Collidable`** (свойства и методы):
- `collision_radius` (свойство, `float`)
- `collides_with(other) -> bool`

**Требования:**
- Все три — с `@runtime_checkable`.
- Все три наследуются от `typing.Protocol`.
- Тела методов — `...` (многоточие).

**Проверьте себя:**
- Класс с методом `draw()` проходит `isinstance(obj, Drawable)`.
- Класс без `draw()` не проходит.
- `42` не проходит `isinstance(42, Drawable)`.

## Задание 3.3. Сохранить и закоммитить

```bash
git add core/game_object.py core/protocols.py
git commit -m "Add GameObject ABC and ability protocols"
```

---
# Шаг 4. Конфигурации через `@dataclass`

## Зачем этот шаг

Магические числа — источник ошибок и головной боли при балансировке. Конфигурации должны быть в одном месте, неизменяемыми и легко заменяемыми.

## Теория: ключевые моменты

1. **`frozen=True`** — для конфигураций, которые не должны меняться. Даёт защиту от случайных присваиваний и делает объект хешируемым.
2. **`field(default_factory=list)`** — для изменяемых значений по умолчанию. Без этого все объекты делили бы один список.
3. **Разделение конфигов.** Общие настройки игры — отдельно, параметры конкретных сущностей — отдельно.

**Для углублённого изучения:**
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [Real Python: Data Classes](https://realpython.com/python-data-classes/)

## Задание 4.1. Аудит магических чисел

Пройдитесь по своему проекту и найдите **все** магические числа. Заполните таблицу:

| Значение | Где встречается | Что означает | В какой конфиг перенести |
|---|---|---|---|
| ... | ... | ... | ... |

**Пример** (для темы «Мистический детектив»):

| Значение | Где | Что означает | Конфиг |
|---|---|---|---|
| 100 | `Detective.__init__` | максимум энергии | `DetectiveConfig.max_energy` |
| 10 | `investigate()` | стоимость осмотра | `GameConfig.investigate_cost` |
| 20 | `use_whip()` | стоимость Плети | `GameConfig.whip_cost` |

## Задание 4.2. Создать конфиги

Создайте файл `core/config.py` с dataclass-классами:

1. **`GameConfig`** (`frozen=True`) — общие настройки (стоимости действий, лимиты).
2. **`<ВашГерой>Config`** (`frozen=True`) — параметры главного героя.
3. **`<ВашНапарник>Config`** (`frozen=True`) — параметры напарника (если есть).
4. **`LocationConfig`** — конфигурация локации (со списками через `field(default_factory=list)`).
5. **`ClueData`** — данные улики/предмета (компонент).

**Проверьте себя:**
- `GameConfig()` — создаётся, значения по умолчанию разумны.
- `cfg.max_energy = 200` → ошибка (frozen).
- Два `LocationConfig` имеют **независимые** списки.

## Задание 4.3. Коммит

```bash
git add core/config.py
git commit -m "Extract magic numbers into dataclass configs"
```

---
# Шаг 5. Миксин логирования

## Зачем этот шаг

Логирование — сквозная функциональность: она нужна многим классам, но не относится ни к одной предметной области. Миксин — идеальный инструмент для неё.

## Теория: правила миксина

1. **Нет `__init__`** — миксин не требует инициализации.
2. **Нет состояния** — только методы. `__slots__ = ()`.
3. **Имя оканчивается на `Mixin`** — соглашение.
4. **Ставится первым** в списке наследования: `class X(LoggingMixin, Base)`.

**Для углублённого изучения:**
- [Python Cookbook: Mixins](https://python-cookbook.readthedocs.io/zh-cn/stable/c08/p18_extending_classes_with_mixins.html)
- [Real Python: Mixins](https://realpython.com/inheritance-composition-python/)

## Задание 5.1. Реализовать `LoggingMixin`

Создайте файл `core/mixins.py`:

**Требования:**

1. Класс `LoggingMixin` с `__slots__ = ()`.
2. Метод `log(message: str) -> None` — логирует на уровне INFO.
3. Метод `log_warning(message: str) -> None` — уровень WARNING.
4. Метод `log_error(message: str) -> None` — уровень ERROR.
5. Использует стандартный модуль `logging`.
6. Формат: `[ИмяКласса] LEVEL: сообщение`.
7. При первом вызове добавляет handler, если его нет.

**Проверьте себя:**

```python
class Foo(LoggingMixin):
    pass

foo = Foo()
foo.log("test")  # должен вывести [Foo] INFO: test
```

## Задание 5.2. Коммит

```bash
git add core/mixins.py
git commit -m "Add LoggingMixin for cross-cutting logging"
```

---
# Шаг 6. Интеграция в существующие классы

## Зачем этот шаг

Мы создали инструменты. Теперь их нужно **применить** к реальным классам проекта. Это самая ответственная часть: нужно не сломать существующую логику.

## Теория: как интегрировать

1. **Порядок наследования:** `class Detective(LoggingMixin, GameObject)`. Миксин первым.
2. **`super().__init__()`** вызывается для `GameObject`, не для миксина.
3. **Реализация абстрактных методов** обязательна. Даже если `update()` пока ничего не делает — метод должен быть.
4. **Использование конфигов** через конструктор: `config`, `game_config`.
5. **Логирование** ключевых действий: создание, изменение состояния, ошибки.

## Задание 6.1. Обновить главный класс героя

Обновите класс `Detective` (или его аналог в вашей теме):

**Требования:**

1. Наследуется от `LoggingMixin` и `GameObject`.
2. В `__init__` принимает `config` и `game_config` (с значениями по умолчанию `None`).
3. Логирует создание: `self.log(f"Создан {name}")`.
4. Все действия используют значения из `GameConfig`, а не магические числа.
5. Логирует ключевые действия: осмотр, атака, отдых, ошибки.
6. Реализует абстрактные методы `update()` и `draw()`.
7. Реализует протокол `Drawable` через `draw()`.

**Проверьте себя:**

```python
det = Detective(config=DetectiveConfig(), game_config=GameConfig())
print(det.draw())
print(isinstance(det, Drawable))  # True
print(isinstance(det, GameObject))  # True
det.investigate("локация")  # в консоли лог
```

## Задание 6.2. Обновить остальные классы

Обновите по той же схеме:
- Напарник (`Div` в примере).
- Локация (`Location`).
- Другие сущности, которые должны быть игровыми объектами.

Для каждой сущности:
- Наследуйте от `GameObject`, если это игровой объект.
- Добавьте `LoggingMixin`, если нужны логи.
- Реализуйте протоколы, которые соответствуют способностям.

## Задание 6.3. Обновить `Game`

Если `Game` создаёт объекты — убедитесь, что он передаёт конфиги в конструкторы. **Не создавайте конфиги внутри классов** — передавайте снаружи (принцип инъекции зависимостей).

## Задание 6.4. Проверить, что игра работает

Запустите игру с `FakeUI` или вручную. Убедитесь, что:
- игра запускается;
- действия логируются;
- значения берутся из конфигов;
- при смене конфига поведение меняется (например, лёгкая сложность).

## Задание 6.5. Коммит

```bash
git add .
git commit -m "Integrate GameObject, protocols, configs and logging into entities"
```

---
# Шаг 7. Юнит-тесты

## Зачем этот шаг

Мы добавили четыре новых инструмента. Каждый имеет **контракт**, который нужно проверить тестами. Без тестов контракты — просто обещания.

## Теория: что тестировать

| Инструмент | Что проверять |
|---|---|
| ABC `GameObject` | Нельзя инстанцировать; подкласс без методов — тоже; полный подкласс работает |
| Протоколы | `isinstance` работает по структуре, а не по наследованию |
| `@dataclass(frozen=True)` | Изменение полей запрещено; равенство по полям |
| `field(default_factory)` | Списки независимы у разных объектов |
| `LoggingMixin` | Работает в любом классе; не имеет состояния |

## Задание 7.1. Тесты для ABC

Создайте `tests/test_game_object.py`. Тесты:

1. `test_cannot_instantiate_abstract` — `GameObject("x")` выбрасывает `TypeError`.
2. `test_subclass_must_implement_methods` — подкласс без `update`/`draw` тоже падает.
3. `test_concrete_subclass_works` — полный подкласс создаётся, имеет уникальный id.
4. `test_unique_ids` — два объекта имеют разные id.
5. `test_deactivate` — после `deactivate()` `active == False`.

## Задание 7.2. Тесты для протоколов

Создайте `tests/test_protocols.py`. Тесты:

1. Объект с `draw()` проходит `isinstance(obj, Drawable)`.
2. Объект без `draw()` не проходит.
3. Объект с `move()` проходит `isinstance(obj, Movable)`.
4. Объект с `collision_radius` и `collides_with` проходит `isinstance(obj, Collidable)`.
5. `42` не проходит `isinstance(42, Drawable)`.

## Задание 7.3. Тесты для dataclass

Создайте `tests/test_config.py`. Тесты:

1. `GameConfig()` — frozen, изменение вызывает ошибку.
2. Значения по умолчанию соответствуют ожиданиям.
3. Два `GameConfig()` равны.
4. Два `LocationConfig` имеют независимые списки.

## Задание 7.4. Тесты для миксина

Создайте `tests/test_mixins.py`. Тесты:

1. `log()` не падает на классе без `__init__`.
2. `log_warning()` и `log_error()` работают.
3. `LoggingMixin.__slots__ == ()`.

## Задание 7.5. Запустить все тесты

```bash
pytest -v
```

**Ожидаемый результат:** все тесты (старые + новые) проходят.

Если что-то падает — исправляйте, пока все не станут зелёными.

## Задание 7.6. Коммит

```bash
git add tests/
git commit -m "Add unit tests for ABC, protocols, dataclass and mixins"
```

---
# Шаг 8. Документация и финальный коммит

## Зачем этот шаг

Документация — часть работы. Без неё ваш проект поймёт только вы (и то через месяц забудете, почему сделали так, а не иначе).

## Задание 8.1. Обновить README

Добавьте в `README.md` разделы:

- **Архитектура** — упомяните ABC `GameObject`, протоколы, конфиги, миксин.
- **Конфигурация** — как создать свой `GameConfig` для лёгкой/сложной игры.
- **Логирование** — как включить/выключить, где искать.

## Задание 8.2. Создать ADR

Создайте `docs/adr_002_game_object.md`:

```markdown
# ADR-002: Введение ABC GameObject и протоколов способностей

## Контекст
<какая проблема была>

## Решение
<что выбрали>

## Рассмотренные альтернативы
1. <альтернатива 1> — почему отклонили
2. <альтернатива 2> — почему отклонили
3. <выбранное решение> — почему выбрали

## Последствия
- ✅ <плюс 1>
- ✅ <плюс 2>
- ⚠️ <минус>
```

## Задание 8.3. Обновить UML

Обновите `docs/uml_module2.md` — итоговая диаграмма должна отражать реальную архитектуру проекта.

## Задание 8.4. Проверка PEP 8

```bash
pycodestyle core/ entities/ game/ ui/ tests/ main.py
```

Исправьте замечания.

## Задание 8.5. Финальный коммит

```bash
git add .
git commit -m "Document module 2: architecture, ADR, UML"
git checkout main
git merge module2-game-objects
```

---
# Самопроверка: чек-лист

In [ ]:
checklist = [
    "ABC GameObject создан и работает",
    "GameObject нельзя инстанцировать напрямую",
    "Подкласс без update/draw не создаётся",
    "Протоколы Movable/Drawable/Collidable созданы",
    "Все протоколы с @runtime_checkable",
    "isinstance работает для протоколов",
    "dataclass-конфиги созданы (GameConfig, DetectiveConfig, ...)",
    "frozen=True использован для конфигов",
    "field(default_factory=list) для списков",
    "LoggingMixin создан и работает",
    "Все игровые сущности наследуют GameObject",
    "Конфиги передаются через конструктор",
    "Логирование подключено к ключевым классам",
    "Игра запускается",
    "Все тесты (старые + новые) проходят",
    "README обновлён",
    "ADR-002 создан",
    "UML обновлён",
    "PEP 8 без замечаний",
    "История коммитов осмысленна",
]
print("Чек-лист для самопроверки:")
for i, item in enumerate(checklist, 1):
    print(f"[ ] {i:2d}. {item}")

---
# Критерии оценки

| Критерий | Баллы | Что проверяется |
|---|---:|---|
| ABC `GameObject` | 1.0 | Контракт, запрет инстанцирования, уникальные id |
| Протоколы | 1.0 | `Movable`/`Drawable`/`Collidable`, `runtime_checkable`, `isinstance` |
| Конфигурации | 1.0 | `frozen=True`, `field(default_factory)`, разумное разделение |
| Миксин логирования | 0.5 | Работает, не имеет состояния, подключается к классам |
| Интеграция | 0.5 | Существующие классы обновлены, игра работает, конфиги передаются |
| Тесты | 0.5 | Покрыты все четыре инструмента, тесты проходят |
| Документация | 0.5 | README, ADR, UML |
| **Итого** | **5.0** | |

**Штрафы:**
- Задержка сдачи: −1 балл за каждую неделю.
- Плагиат из `practice2_game_objects.ipynb`: −2 балла (пример — образец, а не шаблон для копирования).
- Отсутствие тестов: −1 балл.

**Бонусы (до +1):**
- Использование `__slots__` для игровых объектов.
- Реализация дескриптора для валидации атрибутов.
- Дополнительные протоколы (`Serializable`, `EventListener` и т.п.).

---
# Домашнее задание к следующему модулю

1. Завершить работу, смерджить ветку в `main`.
2. Убедиться, что проект работает локально.
3. Прочитать:
   - Martin R. C. «Clean Code», главы о классах.
   - Martin R. C. «Clean Architecture», часть III.
   - [Refactoring Guru: Code Smells](https://refactoring.guru/refactoring/smells)
4. Подготовить вопросы по темам модуля 3:
   - SRP, OCP, LSP, ISP, DIP;
   - Рефакторинг «божественного объекта»;
   - Инверсия зависимостей;
   - Анемичная модель предметной области.

## Что дальше

В модуле 3 мы:
- проведём аудит вашего проекта на нарушение SOLID;
- выделим ответственности в отдельные классы;
- применим инверсию зависимостей;
- подготовим архитектуру к паттернам проектирования.